# Reverse-DCF + Consensus + Multiples — Interactive Notebook

Works for **any listed company** (mega-cap or micro-cap, covered or not).
Runs the full pipeline, shows every chart inline, and lets you tweak assumptions.

**Colab:** upload the whole repo folder (or `!git clone <your repo>`), then run top to bottom.


## 1. Setup

In [ ]:
# In Colab, install deps (skip if running locally with requirements already installed)
# !pip install -q numpy pandas matplotlib
import os
os.environ["VALUATION_OUTPUT_DIR"] = os.path.abspath("outputs")  # write charts next to the notebook
os.makedirs("outputs", exist_ok=True)
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image, display
import run_full_analysis as rfa

## 2. Choose the dataset
`data` = bundled NVDA example. Swap for your own folder of 5 CSVs (see `AGENT_DATA_COLLECTION_PROMPT.md`).

In [ ]:
DATA_DIR = "data"   # or "data_smallcap", or your own folder
res = rfa.run(data_dir=DATA_DIR, n_trials=10_000)

## 3. Master valuation chart

In [ ]:
display(Image(res["charts"]["master"]))

## 4. Consolidated summary table

In [ ]:
res["summary"].style.format({"price": "${:,.2f}"})

## 5. Risk & sensitivity: tornado, WACC×g grid, scenarios

In [ ]:
for key in ["tornado", "grid", "scenarios"]:
    if res["charts"].get(key):
        display(Image(res["charts"][key]))

## 6. Monte Carlo distribution

In [ ]:
display(Image(res["charts"]["monte_carlo"]))

## 7. Consensus & multiples layers (shown only if coverage exists)

In [ ]:
for key in ["bank_targets", "consensus_hist", "multiples", "exit_blend", "hist_mult"]:
    if res["charts"].get(key):
        display(Image(res["charts"][key]))

## 8. Reverse-DCF read-out & backtest

In [ ]:
r = res["reverse"]
print("Market-implied terminal growth:", f"{r['implied_terminal_growth']:.2%}" if r['implied_terminal_growth'] else "n/a")
print("Market-implied WACC:          ", f"{r['implied_wacc']:.2%}" if r['implied_wacc'] else "n/a")
print("Backtest:", res["backtest"])
print("Fair value -> 12m target reconciliation:", res["fv_recon"])

## 9. Tweak assumptions live
Override any driver and re-value without touching the CSVs.

In [ ]:
import copy
from model import ThreeStatementModel
drv = copy.deepcopy(res["drv"]); drv.custom_growth_path = None
drv.year1_growth = 0.30      # <- try your own
drv.wacc          = 0.11
drv.terminal_growth = 0.03
m = ThreeStatementModel(res["base"], drv); m.run()
print(f"Implied price at these assumptions: ${m.dcf_value()['price_per_share']:,.2f} "
      f"(current ${res['base'].current_price:,.2f})")

## 10. One-page HTML dashboard

In [ ]:
import build_dashboard
path = build_dashboard.build(data_dir=DATA_DIR, n_trials=10_000)
print("Open this file in a browser:", path)